In [ ]:
# GLACIER VELOCITY VECTORS CALCULATION

# LICENSE:
# SPDX-License-Identifier: MIT
# Copyright (c)  2026 Natalie Sokalska


# INPUTS: pre-built mesh and bedrock, DSM_files.tif, glacier_outlines.geojson,
# mean temperature values for each year (optional)

# METHODS: Shallow Stream Approximation (SSA) + Shallow Ice Aproximation (SIA), Tikhonov reguralization, Picard iteration

# OUTPUT: raster files with surface, velocity, ice thickness values (.tif)

In [ ]:
# ==========================================
# PHASE 1: IMPORTS / SETUP
# ==========================================
import firedrake                                                    # PDE solver — builds and solves the finite element equations
import icepack                                                      
import geojson                                                      
import numpy as np
import os
import rasterio
from rasterio.transform import from_origin
from scipy.interpolate import NearestNDInterpolator, griddata
from shapely.geometry import shape
from shapely import contains_xy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors



# ==========================================
# 1. LOAD MASTER MESH
# ==========================================
# The mesh and bedrock are pre-computed and stored in an HDF5 checkpoint.
# Replace the filename with your own checkpoint 

with firedrake.CheckpointFile("bedrock_mean.h5",'r') as chk:
    mesh = chk.load_mesh("firedrake_default")                         

print(f"  > Mesh loaded with {mesh.num_cells()} triangles.")


# ==========================================
# 2. DEFINE FUNCTION SPACES
# ==========================================
# CG2 = Continuous Galerkin order 2 (quadratic elements).
# Nodes sit at triangle vertices AND edge midpoints 
Q = firedrake.FunctionSpace(mesh, "CG", 2)                      
V = firedrake.VectorFunctionSpace(mesh, "CG", 2)                

# ==========================================
# 3. CREATE EMPTY DATA CONTAINERS
# ==========================================
h = firedrake.Function(Q, name="Thickness")                     # Ice thickness h [m]
s = firedrake.Function(Q, name="Surface_Elevation")             # Ice surface elevation s [m above sea level]
b = firedrake.Function(Q, name="Bed_Topography")                # Bedrock elevation b [m above sea level]
u = firedrake.Function(V, name="Velocity")                      # Ice velocity vector u [m/yr]

print("--- DONE --- ")


In [ ]:
# ==========================================
# PHASE 2: SURFACE, THICKNESS AND BEDROCK
# ==========================================
# Loads a DEM for the chosen year, smooths it, builds a glacier mask
# from a GeoJSON outline, and computes ice thickness h = s - b.
# The bedrock is loaded from a pre-computed checkpoint (assumed static).


# ==========================================
# 1. USER SETUP  (change these for your glacier / year)
# ==========================================
year_set = "year"                     # number year being simulated   ============================================================================
dem_filename = "dem_<year>.tif" # DEM raster for this year (GeoTIFF, projected CRS in metres)=====================================================


# ==========================================
# 2. LOAD SURFACE (s)
# ==========================================
coords_func = firedrake.Function(V).interpolate(firedrake.SpatialCoordinate(mesh))
coords = coords_func.dat.data_ro

with rasterio.open(dem_filename) as dsm:
    s_raw = icepack.interpolate(dsm, Q)
s_data = s_raw.dat.data

# Identify NoData nodes — DEM pixels that fall outside the raster coverage.
# Adjust the threshold to match your glacier's elevation range.
bad_indices = s_data < 1000.0
good_indices = ~bad_indices

if np.any(bad_indices):
    count = np.sum(bad_indices)
    print(f"  Found {count} 'NoData' values. Filling with Nearest Neighbor")

    good_xy = coords[good_indices, :2]
    good_z  = s_data[good_indices]

    interpolator = NearestNDInterpolator(good_xy, good_z)

    bad_xy = coords[bad_indices, :2]
    filled_values = interpolator(bad_xy)
    s_data[bad_indices] = filled_values

    print(f" Patched {count} holes using nearby data.")
else:
    print("No bad values found.")


# ==========================================
# 3. SMOOTH THE SURFACE
# ==========================================
# Tikhonov smoothing removes DEM noise that would cause unphysical
# thickness spikes and noisy surface gradients in the velocity solver.
# J = 0.5*(s - s_raw)^2 + 0.5*alpha_smooth^2*|grad s|^2
s = firedrake.Function(Q, name="Smoothed_Surface").assign(s_raw)

alpha_smooth = firedrake.Constant(60)
J = 0.5 * (s - s_raw)**2 * firedrake.dx + 0.5 * alpha_smooth**2 * firedrake.inner(firedrake.grad(s), firedrake.grad(s)) * firedrake.dx
firedrake.solve(firedrake.derivative(J, s) == 0, s)
print("  > Surface smoothed ")


# ==========================================
# 4. PREPARE THE MASK
# ==========================================
# k = 1 inside the glacier outline, 0 in the buffer zone.
# The mesh extends beyond the glacier (buffer) so that boundary conditions
# do not interfere with the glacier edge. 
# Replace the GeoJSON path with your glacier outline for the chosen year.

with open("outline_<year>.geojson", 'r') as f:       # Glacier outline for this year (may differ from DEM year if data is sparse) ============================================================================
    mask_json = geojson.load(f)
    mask_polygon = shape(mask_json['features'][0]['geometry'])
print("  > Mask loaded ")

k = firedrake.Function(Q, name=f"Mask_{year_set}")

# all node coordinate
coords_all = coords  

k.dat.data[:] = contains_xy(mask_polygon, coords_all[:, 0], coords_all[:, 1]).astype(float)
print("  > Mask created ")


# ==========================================
# 5. CALCULATE THICKNESS FROM FIXED BEDROCK
# ==========================================
print("calculating thicknes from fixed bedrock")

# Load the bedrock from the precomputed checkpoint.
# Replace the filename and function name with your own.
with firedrake.CheckpointFile("bedrock_mean.h5",'r') as chk:
    b = chk.load_function(mesh, name="Master_Bed_Topography")

# h = s - b: thickness is the height of ice above the bedrock.
# Clamp to [0, max_h]: no negative thickness, upper bound prevents outliers.
h_clamped = firedrake.Function(Q).interpolate(
    firedrake.max_value(0.0, firedrake.min_value(s - b, 400.0))
)
h_clamped.dat.data[:] *= k.dat.data_ro    # zero outside glacier outline

# ── DIAGNOSTIC ────────────────────────────────────────────────
k_vals    = k.dat.data_ro
h_cl_vals = h_clamped.dat.data_ro
inside    = k_vals > 0.5
print(f"  > h inside glacier: min={h_cl_vals[inside].min():.1f} m, "
      f"mean={h_cl_vals[inside].mean():.1f} m, "
      f"max={h_cl_vals[inside].max():.1f} m")
# ─────────────────────────────────────────────────────────────────────────────

h = firedrake.Function(Q, name="Ice_Thickness").assign(h_clamped)

print("  > PHASE 2) DONE")

In [ ]:
# ==============================================================
# PHASE 3: CALCULATING VELOCITY
# Method: Shallow Stream Approximation (SSA) -  Picard iteration,
# implemented directly in Firedrake 
#
# SSA momentum balance:
#   div(h · M)  +  tau_b  =  -rho * g * h * grad(s)
#
# Membrane stress:   M = 2*eta*(eps + tr(eps)*I)
# Glen viscosity:    2*eta = B * (eps_II + eps_0²)^(-1/3),  n = 3
# Schoof-Coulomb:    tau_b = tau_w * tau_max / (tau_w + tau_max)
#                    tau_w = C * |u|^(1/3)  (Weertman sliding)
#                    tau_max = mu * N  (Coulomb yield stress)
#                    N = (1 - lambda_w) * rho * g * h  (effective normal stress)
# ==============================================================
print("--- CALCULATING VELOCITY (SSA Picard) ---")


# ──────────────────────────────────────────────────────────────
# 1. CLIMATE DATA
# Replace with your glacier's mean summer (JJA) temperatures [°C].
# ──────────────────────────────────────────────────────────────
summer_temps = {
    2015: 10.13, #example values used for Belvedere glacier
    2016: 8.75,
    2017: 9.97,
    2018: 9.83,
    2019: 9.88,
    2021: 7.97,
    2022: 10.04,
    2023: 8.78
}
avg_T   = sum(summer_temps.values()) / len(summer_temps)
delta_T = summer_temps[year_set] - avg_T


# ──────────────────────────────────────────────────────────────
# 2. BASE PHYSICS CONSTANTS
# ──────────────────────────────────────────────────────────────
# Glen's fluidity at 0°C 
base_A = 7.5e-17   # Pa^-3 yr^-1

# ice rigidity  B = A^(-1/n),  n = 3   [Pa yr^(1/3)]
B_val = base_A ** (-1.0 / 3.0)

# Weertman friction coefficient [Pa*(yr/m)^(1/3)]
# CALIBRATE: increase >> slower,  decrease >> faster
base_C = 18000


# ──────────────────────────────────────────────────────────────
# 3. CLIMATE MODIFIERS
# A is fixed (temperate glacier)
# lambda_w scales with temperature: warmer >> more meltwater >> lower friction.
# ──────────────────────────────────────────────────────────────
A_val = base_A
C_val = base_C

print(f"  > Summer Temp : {summer_temps[year_set]:.2f} °C  "
      f"(Anomaly: {delta_T:+.2f} °C)")
print(f"  > A = {A_val:.2e} Pa^-3 yr^-1 ")
print(f"  > C = {C_val:.1f} Pa*(yr/m)^(1/3)  ")


# ──────────────────────────────────────────────────────────────
# 4. PREPARE INPUT FIELDS
# ──────────────────────────────────────────────────────────────

# s_ssa: surface for the SSA solver. Ensures h_ssa >= 1 m everywhere.
s_ssa = firedrake.Function(Q, name="Surface_SSA").interpolate(
    firedrake.max_value(s, b + firedrake.Constant(1.0))
)
h_ssa = firedrake.Function(Q, name="Thickness_SSA").interpolate(s_ssa - b)


# Spatially varying C: higher friction at thin-ice margins.
# Increase C_terminus_mult if thin edges are too fast.
h_C_ref         = firedrake.Constant(65)     # [m]
C_terminus_mult = firedrake.Constant(4.0)    # C at h=0 = C_val * C_terminus_mult

C_field = firedrake.Function(Q, name="Friction").interpolate(
    firedrake.Constant(C_val) * (
        C_terminus_mult
        - (C_terminus_mult - firedrake.Constant(1.0))
        * firedrake.min_value(firedrake.Constant(1.0), h_ssa / h_C_ref)
    )
)

# Firedrake constants
rho_ice = firedrake.Constant(917.0)
g_acc   = firedrake.Constant(9.81)
B_cst   = firedrake.Constant(B_val)
eps_0   = firedrake.Constant(1e-3)   # strain-rate regularisation [yr^-1]
eps_u   = firedrake.Constant(0.1)    # velocity regularisation    [m/yr]

# Dirichlet BC: u = 0 at the outer mesh boundary
boundary_ids = list(mesh.exterior_facets.unique_markers)
bcs = firedrake.DirichletBC(
    V, firedrake.Constant([0.0, 0.0]), boundary_ids
)



# ──────────────────────────────────────────────────────────────
# 5. WARM START — SIA sliding estimate
# Picard needs u != 0 .
# Simple Weertman estimate: u ~ (tau_d / C)^3
# ──────────────────────────────────────────────────────────────
grad_s_fd  = firedrake.Function(V).interpolate(firedrake.grad(s_ssa))
slope_fd   = firedrake.Function(Q).interpolate(
    firedrake.sqrt(
        firedrake.inner(firedrake.grad(s_ssa), firedrake.grad(s_ssa)) + 1e-8
    )
)
tau_d_warm = firedrake.Function(Q).interpolate(
    firedrake.min_value(
        rho_ice * g_acc * h_ssa
        * firedrake.min_value(slope_fd, firedrake.Constant(1.0)),
        firedrake.Constant(130000.0)
    )
)
# Weertman sliding estimate, capped at 60 m/yr.
u_speed_warm = firedrake.Function(Q).interpolate(
    firedrake.min_value(
        (tau_d_warm / firedrake.Constant(C_val)) ** 3,
        firedrake.Constant(60.0)
    )
)
u.interpolate(
    -u_speed_warm * grad_s_fd / (slope_fd + firedrake.Constant(1e-10))
)
u.dat.data[:] *= k.dat.data_ro[:, None]   # zero in buffer zone


# ──────────────────────────────────────────────────────────────
# 6. VARIATIONAL FORM  (linearised at u_old)
# ──────────────────────────────────────────────────────────────
u_old   = u.copy(deepcopy=True)
v       = firedrake.TestFunction(V)
u_trial = firedrake.TrialFunction(V)

# Glen viscosity (n=3):  2*eta = B * (eps_e^2 + eps_0^2)^(-1/3)
eps_rate = firedrake.sym(firedrake.grad(u_old))
eps_II   = firedrake.Constant(0.5) * firedrake.inner(eps_rate, eps_rate)
two_eta  = B_cst * (eps_II + eps_0 ** 2) ** firedrake.Constant(-1.0 / 3.0)

# Schoof-Coulomb friction linearised at u_old
# mu:        bed friction angle tangent    lambda_w: pore pressure ratio
# h_wet_ref: thickness at which lambda_w saturates [m]
mu = firedrake.Constant(0.6)
h_wet_ref        = firedrake.Constant(60.0)   # [m]
lambda_w_climate = firedrake.Constant(max(-0.05, min(0.05, delta_T * 0.01)))
lambda_w_base    = firedrake.Constant(0.6)

lambda_w_fn = firedrake.Function(Q, name="PorePressure").interpolate(
    firedrake.min_value(
        firedrake.Constant(1.0),
        (lambda_w_base + lambda_w_climate)
        * firedrake.min_value(firedrake.Constant(1.0), h_ssa / h_wet_ref)
    )
)

N       = (firedrake.Constant(1.0) - lambda_w_fn) * rho_ice * g_acc * h_ssa
tau_max = mu * N

u_II     = firedrake.inner(u_old, u_old)
u_norm   = (u_II + eps_u**2) ** firedrake.Constant(0.5)
tau_w    = C_field * (u_II + eps_u**2) ** firedrake.Constant(1.0/6.0)
tau_coul = tau_w * tau_max / (tau_w + tau_max)
C_eff    = tau_coul / (u_norm + firedrake.Constant(1e-10))

eps_tr = firedrake.sym(firedrake.grad(u_trial))
eps_v  = firedrake.sym(firedrake.grad(v))
dx4    = firedrake.dx(degree=6)

# LHS: membrane stress + basal drag
a = (
    two_eta * h_ssa * (
        firedrake.inner(eps_tr, eps_v)
        + firedrake.div(u_trial) * firedrake.div(v)
    ) * firedrake.dx
    + C_eff * firedrake.inner(u_trial, v) * dx4
)
# RHS: gravitational driving stress
L = (
    -rho_ice * g_acc * h_ssa
    * firedrake.inner(firedrake.grad(s_ssa), v) * firedrake.dx
)

# ──────────────────────────────────────────────────────────────
# 7. PICARD ITERATION
# ──────────────────────────────────────────────────────────────
print("  >  Picard iteration ")

max_iters    = 70
converge_tol =  5e-2
omega        =  0.3      # under-relaxation: damps Picard oscillations

for i in range(max_iters):
    firedrake.solve(
        a == L, u, bcs=bcs,
        solver_parameters={"ksp_type": "preonly", "pc_type": "lu"}
    )
    diff = u.dat.data_ro - u_old.dat.data_ro
    glacier = k.dat.data_ro > 0.5
    res  = np.sqrt((diff ** 2).sum(axis=1))[glacier].max()
    u_old.dat.data[:] = omega * u.dat.data_ro + (1.0 - omega) * u_old.dat.data_ro
    u_old.dat.data[:] *= k.dat.data_ro[:, None]   # zero buffer zone each iteration
    print(f"    Iter {i + 1:2d}:  max delta_u = {res:.4f} m/yr")
    if res < converge_tol:
        print(f"  Converged after {i + 1} iterations.")
        break
else:
    print(f"  Did not fully converge in {max_iters} iterations. "
          f"Using last iterate (acceptable for visualisation).")
u.assign(u_old)

# ──────────────────────────────────────────────────────────────
# 8. APPLY MASK — zero buffer zone velocities
# ──────────────────────────────────────────────────────────────
u.dat.data[:] *= k.dat.data_ro[:, None]

# ──────────────────────────────────────────────────────────────
# 9. ADD SIA INTERNAL DEFORMATION
# SIA adds depth-averaged vertical shear:
#   u_def = -(2A/5) * (rho*g)^3 * |grad(s)|^2 * h^4 * grad(s)
# Uses s_ssa (not raw s). Slope capped at 15% (SIA breaks on steep icefalls).
# ──────────────────────────────────────────────────────────────
slope_sq_raw = (
    firedrake.inner(firedrake.grad(s_ssa), firedrake.grad(s_ssa))
    + firedrake.Constant(1e-8)
)
slope_cap    = firedrake.Constant(0.15)                     # SIA valid for |grad(s)| < cca 0.15
slope_sq_sia = firedrake.min_value(slope_sq_raw, slope_cap ** 2)
scale_sia    = firedrake.sqrt(slope_sq_sia / slope_sq_raw)  # shrinks the vector where slope > cap
grad_s_sia   = firedrake.grad(s_ssa) * scale_sia            # capped gradient: right direction, limited magnitude

u_def = firedrake.Function(V, name="Deformation").interpolate(
    firedrake.Constant(-2.0 * base_A / 5.0)
    * (rho_ice * g_acc) ** 3
    * slope_sq_sia                    # |∇s|^2 (capped)
    * h ** 4                          # h^4
    * grad_s_sia                      # direction: downslope (capped magnitude)
)

u_def.dat.data[:] *= k.dat.data_ro[:, None]          # zero outside glacier
u.dat.data[:] += u_def.dat.data_ro              # total velocity = sliding + deformation


# ──────────────────────────────────────────────────────────────
# 10. DIAGNOSTIC
# ──────────────────────────────────────────────────────────────
u_vals = u.dat.data_ro
speed  = np.sqrt(u_vals[:, 0] ** 2 + u_vals[:, 1] ** 2)

print(f"\n  > Max speed    : {np.nanmax(speed):.1f} m/yr")
print(f"  > Mean speed   : {np.nanmean(speed):.1f} m/yr")
print(f"  > Median speed : {np.nanmedian(speed):.1f} m/yr")
print()
print("  CALIBRATION:")
if np.nanmax(speed) > 40:
    print("  WARNING: Too fast -> increase base_C")
elif np.nanmax(speed) < 10:
    print("  WARNING: Too slow -> decrease base_C")
else:
    print("  OK: Velocity range looks okay.")

print("--- VELOCITY CALCULATED ---")

In [ ]:
# ==========================================
# POST-VELOCITY DIAGNOSTIC - CALIBRATION
# ==========================================
# Compares simulated speed against observed values and suggests a
# new base_C if the model is too fast or too slow.

# ── 1. Extract glacier speeds ───────────────────────────────
u_vals    = u.dat.data_ro
speed_all = np.sqrt(u_vals[:, 0]**2 + u_vals[:, 1]**2)
inside    = k.dat.data_ro > 0.5
speed_in  = speed_all[inside]

# ── 2. Calibration settings ──────────────────────────────────────
# Observed max speeds [m/yr] from field or satellite measurements.
# Add your own years/values.
OBSERVED = {
    #2015: {"max": 27}, #example values for Belvedere glacier
    #2016: {"max": 25}, # replace with your own
    #2017: {"max": 24},
    #2018: {"max": 28},
    #2019: {"max": 31},
}

OUTLIER_CUTOFF_PCT = 98.0
CALIB_PCT          = 95.0

# ── 3. Percentile table ──────────────────────────────────────────
print("  Simulated speed inside glacier (all nodes):")
print(f"  {'Percentile':>12}  {'Speed (m/yr)':>14}")
print("  " + "-" * 30)
for p in [50, 75, 90, 95, 98, 99, 100]:
    v = np.percentile(speed_in, p)
    tag = " <- MAX" if p == 100 else ""
    print(f"  {p:>11}%  {v:>12.1f} m/yr{tag}")

# ── 4. Outlier  ──────────────────────────────
outlier_thresh = np.percentile(speed_in, OUTLIER_CUTOFF_PCT)
n_out          = int((speed_in > outlier_thresh).sum())
speed_clean    = speed_in[speed_in <= outlier_thresh]

print(f"  Outlier nodes : {n_out} ({100*n_out/len(speed_in):.1f}% of glacier nodes) , NOT USED")
print(f"  Clean max     : {speed_clean.max():.1f} m/yr")

v_sim = np.percentile(speed_clean, CALIB_PCT)
print(f"\n  Calibration point ({CALIB_PCT:.0f}th pct of clean glacier): {v_sim:.1f} m/yr")

# ── 5. Calibration result ────────────────────────────────────────
print("\n" + "=" * 60)
print("  CALIBRATION RESULT")
print("=" * 60)

if year_set in OBSERVED:
    v_target      = float(OBSERVED[year_set]["max"])
    ratio         = v_sim / v_target
    C_new         = base_C * (ratio ** (1.0 / 3.0))
    C_new_rounded = round(C_new / 1000) * 1000

    print(f"\n  Observed MAX for {year_set}         : {v_target:.0f} m/yr")
    print(f"  Simulated ({CALIB_PCT:.0f}th pct, clean) : {v_sim:.1f} m/yr")
    print(f"\n  Scaling: C_new = {base_C:.0f} x ({v_sim:.1f}/{v_target:.0f})^(1/3) = {C_new:.0f}")
    print(f"  Rounded suggestion: base_C = {C_new_rounded:.0f}")

    if ratio < 0.95:
        print(f"\n  >> SIMULATED TOO SLOW -- DECREASE base_C to {C_new_rounded:.0f}")
    elif ratio > 1.05:
        print(f"\n  >> SIMULATED TOO FAST -- INCREASE base_C to {C_new_rounded:.0f}")
    else:
        print(f"\n  >> CALIBRATED -- base_C = {base_C:.0f} is good for year {year_set}.")
else:
    print(f"\n  No observed data for year {year_set}.")
    print(f"  Add it to the OBSERVED.")


In [ ]:
# ==========================================
# PHASE 4: EXPORT TO REGULAR GRID FOR BLENDER
# ==========================================
# Interpolates Firedrake fields onto a regular pixel grid and saves as GeoTIFF.
# raster tools (Blender) need a regular grid.

# Read CRS from the source DEM — same projection the mesh was built in
with rasterio.open(dem_filename) as _src:
    mesh_crs = _src.crs
print(f"  > CRS: {mesh_crs}")


# ==========================================
# 1. SETUP THE GRID AND ASPECT RATIO
# ==========================================

# Get the (X, Y) coordinates of every node in the mesh
coords_func = firedrake.Function(V).interpolate(firedrake.SpatialCoordinate(mesh)) 
source_coords = coords_func.dat.data_ro                                      
src_x = source_coords[:, 0]                                                  
src_y = source_coords[:, 1]                                                  

# Bounding box = full mesh extent + 50 m
# Using the same mesh for all years keeps exported TIFFs pixel aligned
min_x, max_x = src_x.min() - 50.0, src_x.max() + 50.0
min_y, max_y = src_y.min() - 50.0, src_y.max() + 50.0

width_m  = max_x - min_x                    # Physical width [m]
height_m = max_y - min_y                    # Physical height [m]

# Fix pixel width at 1024 (can be adjusted), compute height to keep square pixels
RES_X = 1024
aspect_ratio = height_m / width_m
RES_Y = int(RES_X * aspect_ratio) 

print(f"  > Physical Size: {width_m:.1f}m x {height_m:.1f}m")
print(f"  > Grid defined: {RES_X}x{RES_Y} pixels (Aspect Ratio: {aspect_ratio:.3f}).")

# Regular grid 
grid_x, grid_y = np.meshgrid(                                                
    np.linspace(min_x, max_x, RES_X),                                        
    np.linspace(min_y, max_y, RES_Y)                                         
)


# ==========================================
# 2. DEFINE EXPORT FUNCTION
# ==========================================
def safe_export_tiff(values, filename, fill_val=0.0, crs=None):
    print(f"    Processing {filename}")
    grid_z = griddata(
        (src_x, src_y), values, (grid_x, grid_y),
        method='linear', fill_value=fill_val,
    )
    grid_z_flipped = np.flipud(grid_z)   # image rows go top-to-bottom
    pixel_size_x = (max_x - min_x) / RES_X
    pixel_size_y = (max_y - min_y) / RES_Y
    transform = from_origin(min_x, max_y, pixel_size_x, pixel_size_y)
    with rasterio.open(
        filename, 'w',
        driver='GTiff',
        height=RES_Y, width=RES_X,
        count=1, dtype='float32',
        crs=crs, transform=transform,
    ) as dst:
        dst.write(grid_z_flipped.astype('float32'), 1)
    print(f" Saved: {filename}")


# ==========================================
# 3. RUN EXPORT
# ==========================================
output_dir = f"Blender_Input_{year_set}"                                     
if not os.path.exists(output_dir):                                           
    os.makedirs(output_dir)                                                  

try:
    # SURFACE (s)
    # fill_val = min elevation so areas outside the glacier are flat, not holes
    s_vals = s.dat.data_ro                                                   
    min_elev = np.nanmin(s_vals)
    safe_export_tiff(s_vals, f"{output_dir}/SURFACE_{year_set}.tif", fill_val=min_elev, crs=mesh_crs)

    
    # BEDROCK (b)
    b_vals = b.dat.data_ro
    safe_export_tiff(b_vals, f"{output_dir}/BED_{year_set}.tif", fill_val=np.nanmin(b_vals), crs=mesh_crs)

    
    # THICKNESS (h)
    h_vals = h.dat.data_ro                                                              
    safe_export_tiff(h_vals, f"{output_dir}/THICKNESS_{year_set}.tif", fill_val=0.0, crs=mesh_crs)

    
    # VELOCITY (u) 
    # FLOW tiff has 3 channels: Red = Vx [m/yr], Green = Vy [m/yr], Blue = unused (zero)
    u_vals = u.dat.data_ro                                                   
    
    print("    Interpolating Velocity X")                                 
    grid_u_x = griddata((src_x, src_y), u_vals[:,0], (grid_x, grid_y), method='linear', fill_value=0)
    
    print("    Interpolating Velocity Y")                                
    grid_u_y = griddata((src_x, src_y), u_vals[:,1], (grid_x, grid_y), method='linear', fill_value=0)
    
    grid_u_combined = np.dstack((grid_u_x, grid_u_y, np.zeros_like(grid_u_x)))  
    
    pixel_size_x = (max_x - min_x) / RES_X                                   
    pixel_size_y = (max_y - min_y) / RES_Y                                   
    transform = from_origin(min_x, max_y, pixel_size_x, pixel_size_y)        
    
    grid_u_combined = np.flipud(grid_u_combined)                             # Flip Y for image format
    
    with rasterio.open(f"{output_dir}/FLOW_{year_set}.tif", 'w', driver='GTiff',
                       height=RES_Y, width=RES_X, count=3, dtype='float32',
                       crs=mesh_crs, transform=transform) as dst:
        dst.write(grid_u_combined[:,:,0].astype('float32'), 1)               # Red   = Vx
        dst.write(grid_u_combined[:,:,1].astype('float32'), 2)               # Green = Vy
        dst.write(grid_u_combined[:,:,2].astype('float32'), 3)               # Blue  = 0 (unused)
    print(f" Saved: {output_dir}/FLOW_{year_set}.tif")

except Exception as e:
    print(f"Error during export: {e}")

In [ ]:
# ==========================================
# EXPORT RAW (UNSMOOTHED) SURFACE FOR BLENDER
# ==========================================
# Exports s_raw (DEM before Tikhonov smoothing) as a GeoTIFF
# with the same grid size, bounding box, and CRS as the other Blender exports.

s_raw_vals = s_raw.dat.data_ro
min_elev_raw = np.nanmin(s_raw_vals)
safe_export_tiff(s_raw_vals, f"{output_dir}/SURFACE_RAW_{year_set}.tif",
                 fill_val=min_elev_raw, crs=mesh_crs)


In [ ]:
print("--- PLOTTING VELOCITY VECTORS (fixed colour scale 0-35 m/yr, black background) ---")

# ── EXTRACT DATA ────────────────────────────────────────────
X_expr   = firedrake.SpatialCoordinate(V.mesh())
f_coords = firedrake.Function(V).interpolate(X_expr)

x_coords = f_coords.dat.data_ro[:, 0]
y_coords = f_coords.dat.data_ro[:, 1]

u_x   = u.dat.data_ro[:, 0]
u_y   = u.dat.data_ro[:, 1]
speed = np.sqrt(u_x**2 + u_y**2)

# ── FIXED COLOUR SCALE ─────────────────────────────────────
VMIN = 0.0    # m/yr
VMAX = 35.0   # m/yr  (change if needed)
norm = mcolors.Normalize(vmin=VMIN, vmax=VMAX)

# ──  ARROWS ────────────────────────────────────────
step      = 80
x_sub     = x_coords[::step]
y_sub     = y_coords[::step]
u_x_sub   = u_x[::step]
u_y_sub   = u_y[::step]
speed_sub = speed[::step]

# ───────────────────────────
with plt.style.context('dark_background'):
    fig, ax = plt.subplots(figsize=(14, 12))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    quiv = ax.quiver(
        x_sub, y_sub,
        u_x_sub, u_y_sub,
        speed_sub,
        cmap='jet',
        norm=norm,
        scale=350,
        width=0.0015,
        pivot='mid',
        clim=(VMIN, VMAX),
    )

    cbar = fig.colorbar(quiv, ax=ax, label="Speed (m/yr)", extend='max')
    cbar.set_ticks([0, 5, 10, 15, 20, 25, 30, 35])
    cbar.set_label("Speed (m/yr)", color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    ax.set_title(
        f"Glacier Flow Vectors  |  Year {year_set} | Max: {speed.max():.1f} m/yr\n"
        f"Colour scale: {VMIN}–{VMAX} m/yr  "
        f"(arrows above {VMAX} m/yr clipped to top colour)\n"
        f"1 arrow per {step} nodes  |  base_C = {base_C}\n"
        f"base_A={base_A} | h_C_ref= {float(h_C_ref)} | C_terminus_mult= {float(C_terminus_mult)}\n\n\n\n",
        fontsize=12, color='white'
    )
    ax.set_xlabel("Easting", color='white')
    ax.set_ylabel("Northing", color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')
    ax.set_aspect('equal')
    plt.tight_layout()

    # ── 5. SAVE ────────────────────────────────────────────────
    output_filename = f"Glacier_Velocity_Vectors_BLACK_{year_set}.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight',
                facecolor='black')
    print(f"  > Saved: {output_filename}")
    plt.show()
